In [1]:
!nvidia-smi

Sat Sep 26 15:59:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/hannna24/Github-IssueHawk.git

Cloning into 'Github-IssueHawk'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 119 (delta 42), reused 107 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.53 MiB | 14.38 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [5]:
%cd Github-IssueHawk

/content/Github-IssueHawk


In [6]:
!ls

backend  mine.txt   requirements.txt  run_baseline.py
data	 README.md  requirments.txt   training


In [7]:
!ls data/splits

test.jsonl  train.jsonl  val.jsonl


In [8]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 123.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 51.3 MB/s eta 0:00:00


In [9]:
%cd /content/Github-IssueHawk/training
from prepare_dataset import build_dataset
ds = build_dataset('../data/splits/train.jsonl')
print(ds)
print('\n--- one formatted example ---\n')
print(ds[0]['text'])

/content/Github-IssueHawk/training


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Dataset({
    features: ['text'],
    num_rows: 1639
})

--- one formatted example ---

<|im_start|>system
You are an issue triage expert for open-source repositories. Classify the issue into EXACTLY ONE label: bug, feature-request, documentation. Respond with ONLY JSON: {"label": "<label>"}.<|im_end|>
<|im_start|>user
ISSUE TITLE: BUG: Subsetting a df can sometimes change pandas behaviour into thinking the removed values still exists
ISSUE BODY: ### Pandas version checks

- [X] I have checked that this issue has not already been reported.

- [X] I have confirmed this bug exists on the [latest version](https://pandas.pydata.org/docs/whatsnew/index.html) of pandas.

- [X] I have confirmed this bug exists on the [main branch](https://pandas.pydata.org/docs/dev/getting_started/install.html#installing-the-development-version-of-pandas) of pandas.


### Reproducible Example

```python
import pandas as pd
import numpy as np

# Create random data
data = {"Group": ["A"] * 3 + ["B"] * 3 + ["C"]


Those <|im_start|> / <|im_end|> markers are Qwen's chat-template special tokens, applied automatically. This is precisely what the model needs to learn from

## A. Set location & imports
Make sure we're in the `training/` folder (so `from prepare_dataset import ...` works) and load every library we need.



In [11]:
%cd /content/Github-IssueHawk/training

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from prepare_dataset import build_dataset

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
print('imports OK')

/content/Github-IssueHawk/training
imports OK


## B. Detect GPU precision (the T4 fix)
A T4 has no bf16 hardware, so this auto-picks **fp16**. You should see `using fp16 (correct for a T4)`.



In [12]:
BF16_OK = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'using {"bf16" if BF16_OK else "fp16"} (correct for a T4)')

GPU: Tesla T4
using bf16 (correct for a T4)


## C. Load the base model in 4-bit
Downloads Qwen2.5-3B (~a few GB, **one-time**) and loads it in 4-bit so it fits the T4.
Slowest cell the first time — a few minutes. Ignore any `HF_TOKEN` warning; it's harmless.


In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,   # float16 on a T4
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto'
)
print('\n✅ base model loaded in 4-bit')

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ base model loaded in 4-bit


## D. Attach the LoRA adapters
Freezes the base model and injects small trainable adapters into the attention layers.
The printout should show **trainable params well under 1%** — that's LoRA working.


In [14]:
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()   # essential on a free GPU

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


## E. Build the train & validation datasets
Formats your real Layer 1 splits into chat-template examples. You saw this work already (1639 rows).


In [15]:
train_ds = build_dataset('../data/splits/train.jsonl')
val_ds   = build_dataset('../data/splits/val.jsonl')
print(f'train: {len(train_ds)} examples   val: {len(val_ds)} examples')

train: 1639 examples   val: 351 examples


## F. Configure the trainer (T4-safe)
All the training settings. The `try/except` blocks handle newer-vs-older versions of the `trl`
library automatically (the renamed `max_length` and `processing_class`), so you won't hit a `TypeError`.


In [ ]:
# Mount Drive so checkpoints survive a Colab disconnect (idempotent — safe to re-run)
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/issuehawk-checkpoints/lora-adapter'

cfg = dict(
    output_dir=CKPT_DIR,               # save straight to Drive, not ephemeral /content
    num_train_epochs=3,
    per_device_train_batch_size=2,     # T4-safe (doc used 4)
    gradient_accumulation_steps=8,     # 2 x 8 = effective batch 16
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy='steps', eval_steps=50,
    save_strategy='steps',             # save mid-epoch, not just at epoch boundaries
    save_steps=25,                     # ~4 saves per epoch (worst-case loss = 25 steps)
    save_total_limit=3,                # keep only the 3 newest to bound Drive usage
    bf16=BF16_OK, fp16=not BF16_OK,    # fp16 on a T4
    report_to='none',                  # no wandb login needed
    dataset_text_field='text',
)
try:
    sft_config = SFTConfig(max_length=1024, **cfg)      # newer trl
except TypeError:
    sft_config = SFTConfig(max_seq_length=1024, **cfg)  # older trl

try:
    trainer = SFTTrainer(model=model, args=sft_config,
                         train_dataset=train_ds, eval_dataset=val_ds,
                         processing_class=tokenizer)     # newer trl
except TypeError:
    trainer = SFTTrainer(model=model, args=sft_config,
                         train_dataset=train_ds, eval_dataset=val_ds,
                         tokenizer=tokenizer)            # older trl
print('✅ trainer ready')

## G. Train  ▶️  *(the long one — ~30–90 min)*
A progress bar appears and loss values print as it goes. **A steadily falling loss is healthy.**

Checkpoints are saved every 25 steps **to Google Drive**, so if Colab disconnects you can just re-run this cell — it detects the existing checkpoint and resumes from where it stopped.

In [ ]:
import os
# If a checkpoint already exists in Drive from a previous (interrupted) run, resume from it.
resume = os.path.isdir(CKPT_DIR) and any(
    d.startswith('checkpoint-') for d in os.listdir(CKPT_DIR)
)
print(f'{"▶️  Resuming" if resume else "🆕 Starting fresh"} — output dir: {CKPT_DIR}')

trainer.train(resume_from_checkpoint=resume)
trainer.save_model(CKPT_DIR)
print(f'\n✅ Adapter saved to {CKPT_DIR}')